# Evolution at an invasion front

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/zoo/evolving_front.ipynb)

*Syga S, Nava-Sedeño JM, Deutsch A (2026). A novel cellular automaton
approach for modeling genotypic and phenotypic heterogeneity in cell
systems. Eur Phys J Spec Top.
[doi:10.1140/epjs/s11734-026-02186-1](https://doi.org/10.1140/epjs/s11734-026-02186-1)*

**Question.** A population invades empty space, and every cell passes its
proliferation rate on to its daughters, with small mutations. How fast does
the population spread, and where do the fastest-dividing cells end up?

The paper introduces the *evolutionary LGCA*: cells carry individual
properties, as in identity-based LGCA, but any number of them may share a
channel, as in LGCA without volume exclusion. In BioLGCA this is an
identity-based model with `volume_exclusion=False`. The notebook reproduces
Fig 3: an invasion wave whose front accelerates as the fastest cells gather
at it.

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import lgca
from lgca import zoo
from lgca.zoo import evolving_front as front

zoo.parameter_table("evolving_front")

## The model

Every cell has its own proliferation rate $\alpha$. In every time step:

1. **Birth and death:** every cell dies with probability $\delta$ and,
   independently, divides with probability $\alpha(1 - n/K)$, where $n$ is
   the number of cells at its node and $K$ the carrying capacity.
2. **Mutation:** a daughter's $\alpha$ mutates with probability $p_\mu$ to
   a value drawn from a normal distribution around the mother's, with
   variance $\sigma^2 = 10^{-4}$ (kept within $[0, 1]$).
3. **Migration:** every cell takes a channel at random, a velocity channel
   with weight 1 or the rest channel with weight $e^\gamma$. A cell thus
   moves with probability $b/(b + e^\gamma)$, and $\gamma$ sets the diffusion
   coefficient $D$ (here $D = 0.14$).

On a 500 × 10 strip the cells start at the left edge with $\alpha_0 = 0.2$
at the equilibrium density $\Psi_0 = 1 - \delta/\alpha_0$. The mean-field
limit is a Fisher–KPP equation,
$\partial_t\Psi = D\,\partial_x^2\Psi + \bar\alpha\,\Psi(1 - \Psi) - \delta\Psi$,
whose fronts move at

$$v(\bar\alpha) = 2\sqrt{D(\bar\alpha - \delta)}.$$

As $\bar\alpha$ evolves, so does the speed.

In [ ]:
spec = front.build_spec()
for operator in spec.dynamics.operators:
    print(operator)
record = front.record(spec, every=10)  # 1000 steps, about 25 s

## Fig 3: the wave, its front and the evolving proliferation rate

In [ ]:
steps, x_0 = record["steps"], record["front"][0]
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), constrained_layout=True)

extent = (-0.5, record["density"].shape[1] - 0.5, steps[-1], 0)
image = axes[0, 0].imshow(record["density"], aspect="auto", extent=extent, cmap="Greys", interpolation="nearest")
fig.colorbar(image, ax=axes[0, 0], label="cells per node")
for alpha, style in ((0.2, ":"), (record["alpha_mean"][-1], "--")):  # fronts at a constant speed
    axes[0, 0].plot(x_0 + front.wave_speed(alpha) * steps, steps, "r", ls=style, lw=1)
axes[0, 0].set(xlim=extent[:2], ylim=extent[2:], xlabel="$x$", ylabel="time step $k$", title="(a) density")

axes[0, 1].plot(steps, record["front"], "k", lw=2, label="simulation")
K = spec.state.capacity
for alpha, label, style in ((0.2, "α₀", ":"), (record["alpha_mean"], "population mean", "--"),
                            (record["alpha_top"], "fastest 10 %", "-.")):
    axes[0, 1].plot(steps, front.predicted_front(steps, alpha, x_0, capacity=K), "0.4", ls=style,
                    label=f"Fisher–KPP with {label}")
axes[0, 1].set(xlabel="time step $k$", ylabel="front position", title="(b) front")
axes[0, 1].legend()

image = axes[1, 0].imshow(record["alpha"], aspect="auto", extent=extent, cmap="Greys", interpolation="nearest")
fig.colorbar(image, ax=axes[1, 0], label="mean α")
axes[1, 0].set(xlabel="$x$", ylabel="time step $k$", title="(c) mean proliferation rate")

mean, std = record["alpha_mean"], record["alpha_std"]
axes[1, 1].plot(steps, mean, "k")
axes[1, 1].fill_between(steps, mean - std, mean + std, color="grey", alpha=0.3)
axes[1, 1].set(xlabel="time step $k$", ylabel="α (mean ± sd)", title="(d) proliferation rate of all cells")
plt.show()
plt.close(fig)

print(f"discreteness correction 1 - 4/ln²K = {front.discreteness(K):.2f}; rms distance from the simulated front:")
for alpha, label in ((0.2, "α₀"), (record["alpha_mean"], "population mean"), (record["alpha_top"], "fastest 10 %")):
    errors = [np.sqrt(np.mean((front.predicted_front(steps, alpha, x_0, capacity=c) - record["front"]) ** 2))
              for c in (K, None)]
    print(f"  Fisher–KPP with {label}: {errors[0]:5.1f} nodes corrected, {errors[1]:5.1f} without the correction")

As in the paper:

- **(a)** The population invades as a wave, and the front curves: it
  accelerates. The red lines are fronts at a constant speed, for the
  initial $\alpha_0$ (dotted) and for the final mean $\bar\alpha$ (dashed).
- **(c)** The fastest-dividing cells sit at the front, the slower ones
  behind it, near the left edge: a gradient of proliferation rates builds
  up along the wave. Cells at the front divide into empty space, so their
  lineages grow and carry their high $\alpha$ forward; behind the front
  the node is full and nobody divides much.
- **(d)** The mean proliferation rate and its spread grow over time.

**(b)** The front is predicted best with the mean $\alpha$ of the fastest
10 % of the cells, as in the paper: the cells that carry the front are the
fast ones. The predictions are corrected for the discreteness of the front,
as in the paper's code: at the leading edge there are only a few cells,
and a front of discrete cells moves more slowly than the continuum front,
here by the factor $1 - 4/\ln^2 K = 0.81$ (after Brunet and Derrida).
Without the correction the population mean happens to predict the front
best, and the fastest 10 % overshoot: the correction is as large as the
effect it is meant to reveal. Without mutation, at a fixed
$\alpha_0 = 0.2$, the front here moves at 0.29 nodes per step against
$v(\alpha_0) = 0.33$, 11 % slower, a smaller slow-down than the factor
0.81 assumes.

## Differences from the paper

- **K and the run length**, $K = 100$ and 1000 steps (until the front
  reaches the right edge), are those of the paper's code; its text does
  not give them.
- **Diffusion.** $\gamma = \ln(b/4D - b)$ gives $D = 0.14$: a cell moves
  with probability $b/(b + e^\gamma) = 4D$ and then one node, so its
  displacement along an axis has variance $2D$ per step.
- **The front** is the first $x$ where the density falls below
  $0.1\,\Psi_0$, as in the paper's text.

## Explore it

A shorter strip, with the mean $\alpha$ per node as the view. Change the
mutation size $\sigma$ or the death rate $\delta$ while the wave runs:

In [ ]:
explorer = lgca.explore(
    front.build_spec(length=200, capacity=50),
    {"dynamics.operators[birth_death].mutation.traits.r_b.scale": (0.0, 0.05),  # σ
     "death_rate": (0.0, 0.15)},  # δ
    view="mean r_b",
    measure=["population", "r_b"],
    steps_per_frame=5,
)
explorer

## Things to try

1. **No mutation** ($p_\mu = 0$ or $\sigma = 0$): does the front keep a
   constant speed $v(\alpha_0)$?
2. **Well mixed.** Replace the strip by a small periodic lattice: does
   $\bar\alpha$ still rise, and as fast? Selection at the front is spatial.
3. **Death.** Raise $\delta$ towards $\alpha_0$. The wave slows, and for
   $\delta > \alpha_0$ the population can only survive if faster mutants
   appear in time.
4. **Discreteness.** Measure the front speed at a fixed $\alpha$
   ($p_\mu = 0$) for $K$ from 10 to 1000 and compare it with
   $2\sqrt{D(\alpha - \delta)}$. How does the gap shrink with $K$?